# A generic branch-and-bound algorithm for l0-regularized problems.

*C. Elvira, T. Guyard, and C. Herzet. Submitted in 2025.*

#### Notebook to reproduce experiments of Section 6.1

---

This experiment compares the performance of different methods to fit a regularization path for on L0-regularized problems with real-worlds data. To reproduce it, let first import all the necessary packages and routines.

In [2]:
from el0ps.datafit import Leastsquares, Logistic, Squaredhinge
from el0ps.penalty import BigmL1L2norm, BigmL2norm
from el0ps.path import Path
from el0ps.utils import compute_lmbd_max
from l0exp.dataset import get_dataset_realworld
from l0exp.solver import get_solver, can_handle_instance

Now, let select the dataset to use. Each dataset contains a feature matrix `A` and a target vector `y`. Possible choices are:
- `dexter`: Text classification problem from the NIPS 2003 feature selection challenge.
- `dorothea`: Drug discovery problem from the NIPS 2003 feature selection challenge.
- `drug`: Drug discovery problem from "DeepCDR: a hybrid graph convolutional network for predicting cancer drug response" by Quiao Liu et al. (2020).
- `gisette`: Handwritten digit recognition problem from the NIPS 2003 feature selection challenge.
- `kits`: Kidney tumor classification dataset from the KiTS19 challenge.
- `madelon`: Synthetic dataset from the NIPS 2003 feature selection challenge.
By default, datasets are normalized.

In [3]:
dataset = "kits"
A, y, x = get_dataset_realworld(name=dataset, normalize=True)
print(f"Dataset: {dataset}")
print(f"A shape: {A.shape}")
print(f"y shape: {y.shape}")

Dataset: kits
A shape: (100, 27648)
y shape: (100,)


We can next define the problem data-fidelity and penalty functions, as well as the L0-norm weight involved in the L0-regularized problem. In our experiments, the following values have been used:

| Dataset   | Data-fidelity            | Penalty                                                                  | L0-norm weight `lmbd`   |
|-----------|--------------------------|--------------------------------------------------------------------------|-------------------------|
| dexter    | `Squaredhinge(y)`        | `BigmL1L2norm(M=5.4787, alpha=0.03, beta=0.03)`                          | `0.16263408333333365`   |
| dexter    | `Squaredhinge(y)`        | `BigmL2norm(M=1.5573, beta=0.03)`                                        | `1.0305636626509835`    |
| dorothea  | `Squaredhinge(y)`        | `BigmL1L2norm(M=1.6547, alpha=0.08, beta=0.201)`                         | `0.07362415091792515`   |
| dorothea  | `Squaredhinge(y)`        | `BigmL2norm(M=2.8971, beta=0.08)`                                        | `0.1861538461538452`    |
| drug      | `Leastsquares(y)`        | `BigmL1L2norm(M=6.0007, alpha=1.3028, beta=0.2065)`                      | `1.8703554108838927`    |
| drug      | `Leastsquares(y)`        | `BigmL2norm(M=5.3828, beta=0.5186)`                                      | `6.388764414246429`     |
| gisette   | `Logistic(y)`            | `BigmL1L2norm(M=3.7974, alpha=0.6, beta=0.6)`                            | `0.014622543905747125`  |
| gisette   | `Logistic(y)`            | `BigmL2norm(M=3.6407, beta=0.6)`                                         | `0.015574214944236539`  |
| kits      | `Leastsquares(y)`        | `BigmL1L2norm(M=0.1402, alpha=0.01, beta=1.0)`                           | `0.0019604069640045078` |
| kits      | `Leastsquares(y)`        | `BigmL2norm(M=0.1352, beta=1.0)`                                         | `0.001967430593400572`  |
| madelon   | `Logistic(y)`            | `BigmL1L2norm(M=0.6139, alpha=0.2, beta=1.2619)`                         | `0.00015110927258995388`|
| madelon   | `Logistic(y)`            | `BigmL2norm(M=0.7105, beta=1.2619)`                                      | `3.345450872080412e-05` |

In [5]:
datafit = Leastsquares(y)
penalty = BigmL1L2norm(M=0.1402, alpha=0.01, beta=1.0)
lmbd    = 0.0019604069640045078
print(f"datafit   : {datafit}")
print(f"penalty   : {penalty}")
print(f"lambda    : {lmbd}")
print(f"lambda_max: {compute_lmbd_max(datafit, penalty, A)}")

datafit   : Leastsquares
penalty   : BigmL1L2norm
lambda    : 0.0019604069640045078
lambda_max: 0.7636854461377822


Finally, we can define the different methods to compare and their parameters. Here is the setup used in our experiments. Solvers that cannot handle the considered instance are automatically skipped.

In [ ]:
# Solvers to use (comment out those you don't want to run or that are not installed)
solver_types = [
    "el0ps",
    "l0bnb",
    "mimosa",
    "gurobi",
    "mosek",
    "oa"
]

solver_args = {
    "time_limit"    : 3600,   # time limit in seconds for each value of lambda in the path
    "relative_gap"  : 1.e-8,  # relative optimality gap on the objective value
    "verbose"       : False,  # verbosity toggle for solvers
}

path_args = {
    "lmbd_max"          : 1.0,      # maximum value of lmbd / lmbd_max in the path
    "lmbd_min"          : 0.01,     # minimum value of lmbd / lmbd_max in the path
    "lmbd_num"          : 20,       # number of values of lmbd in the path
    "lmbd_normalized"   : True,     # normalize lmbd w.r.t lmbd_max in the path
}

for solver_type in solver_types:

    try:    
        solver = get_solver(solver_type, solver_args)
        if can_handle_instance(solver, datafit, penalty):
            print(f"Running {solver_type}...")
            path = Path(**path_args)
            path.fit(solver, datafit, penalty, A)
        else:
            print(f"Skipping {solver_type}: cannot handle this instance")
    except Exception as e:
        print(f"Solver {solver_type} failed with error: {e}")
    print()

Running el0ps...
------------------------------------------------------------
      lambda      status        time   objective     num nnz
------------------------------------------------------------
        0.76     optimal      3.1751       27.50           0
        0.60     optimal      0.6022       26.80           9
        0.47     optimal      1.2741       25.26          15
        0.37     optimal      0.5863       23.43          21
        0.29     optimal      0.2756       21.61          25
        0.23     optimal      2.7643       19.95          28
        0.18     optimal      0.6119       18.49          31
        0.14     optimal      0.6023       17.26          33
        0.11     optimal      1.4355       16.24          35
        0.09     optimal      0.7745       15.40          36
        0.07     optimal      1.1578       14.73          37
        0.05     optimal      2.3825       14.07          55
        0.04     optimal      8.1283       13.32          77
